# Prepare the chest X-ray dataset for a CNN

This notebook downloads and extracts the Kaggle dataset, keeps its official test set, makes a reproducible stratified train/validation split, copies the images into `train`, `val`, and `test` folders, and displays labeled samples.

The Kaggle URL in the request appears shortened; this notebook uses the canonical dataset slug `paultimothymooney/chest-xray-pneumonia`. Kaggle may require you to sign in or configure your Kaggle API credentials before downloading. The image-copy step needs extra disk space (roughly another copy of the dataset).

In [ ]:
%pip install -q kagglehub scikit-learn matplotlib Pillow tensorflow

## Download and extract

`kagglehub.dataset_download` downloads and extracts the dataset, then returns its local path. This cell looks for the expected `train` and `test` folders, including when they are nested under `chest_xray`.

In [ ]:
from pathlib import Path
import kagglehub

DATASET_SLUG = "paultimothymooney/chest-xray-pneumonia"
SEED = 42

download_dir = Path(kagglehub.dataset_download(DATASET_SLUG))
print(f"Downloaded/extracted dataset to: {download_dir}")

candidates = [download_dir]
candidates.extend(path for path in download_dir.rglob("*") if path.is_dir())
data_root = next(
    (
        path
        for path in candidates
        if (path / "train").is_dir() and (path / "test").is_dir()
    ),
    None,
)
if data_root is None:
    raise FileNotFoundError(
        f"Could not find folders named 'train' and 'test' under {download_dir}. "
        "Check that the Kaggle dataset downloaded and extracted correctly."
    )

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}
class_names = sorted(
    path.name for path in (data_root / "train").iterdir() if path.is_dir()
)
if not class_names:
    raise FileNotFoundError(f"No class folders found in {data_root / 'train'}")

print(f"Data root: {data_root}")
print(f"Classes: {class_names}")

## Make stratified splits and copy images

The dataset's official `test` folder stays untouched. The notebook combines the original `train` and (if present) `val` images, then stratifies a validation subset by class. The source test set is already close to 10% of this dataset, so the resulting proportions are approximately 80/10/10. Re-running this cell with the same seed produces the same split.

In [ ]:
import shutil
from collections import Counter
from sklearn.model_selection import train_test_split

def collect_images(split_names):
    records = []
    for split_name in split_names:
        split_dir = data_root / split_name
        if not split_dir.is_dir():
            continue
        for class_name in class_names:
            class_dir = split_dir / class_name
            if class_dir.is_dir():
                records.extend(
                    (image_path, class_name)
                    for image_path in class_dir.rglob("*")
                    if image_path.is_file()
                    and image_path.suffix.lower() in IMAGE_EXTENSIONS
                )
    return records

training_pool = collect_images(["train", "val"])
test_records = collect_images(["test"])
if not training_pool or not test_records:
    raise FileNotFoundError(
        "No images were found in the expected class folders. "
        f"Training images: {len(training_pool)}, test images: {len(test_records)}."
    )

train_records, val_records = train_test_split(
    training_pool,
    test_size=0.10,
    random_state=SEED,
    stratify=[label for _, label in training_pool],
)
splits = {"train": train_records, "val": val_records, "test": test_records}

output_dir = download_dir.parent / "chest_xray_cnn_splits"
for split_name, records in splits.items():
    for source_path, class_name in records:
        destination_dir = output_dir / split_name / class_name
        destination_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_path, destination_dir / source_path.name)

print(f"\nSplit folders: {output_dir}")
for split_name, records in splits.items():
    counts = Counter(label for _, label in records)
    print(f"{split_name:>5}: {len(records):,} images | " + ", ".join(
        f"{name}: {counts.get(name, 0):,}" for name in class_names
    ))

## Check for duplicate images

This checks the generated split folders for byte-identical image files using SHA-256. It reports duplicates within a split and across splits (which can cause data leakage); it does not delete or move any files. Re-encoded or resized copies may not have identical file hashes.

In [ ]:
import hashlib
from collections import defaultdict

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as image_file:
        for chunk in iter(lambda: image_file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

hash_to_files = defaultdict(list)
split_order = {"train": 0, "val": 1, "test": 2}
for split_name in ("train", "val", "test"):
    split_dir = output_dir / split_name
    if not split_dir.is_dir():
        continue
    for image_path in split_dir.rglob("*"):
        if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
            hash_to_files[sha256_file(image_path)].append(image_path)

if not hash_to_files:
    raise FileNotFoundError(f"No images found in split folders under {output_dir}")

duplicate_groups = [paths for paths in hash_to_files.values() if len(paths) > 1]
cross_split_groups = [
    paths
    for paths in duplicate_groups
    if len({path.relative_to(output_dir).parts[0] for path in paths}) > 1
]
removed_count = 0
print(f"Unique image files checked: {sum(map(len, hash_to_files.values())):,}")
print(f"Exact duplicate groups before removal: {len(duplicate_groups):,}")
print(f"Duplicate groups spanning splits before removal: {len(cross_split_groups):,}")

for group_number, paths in enumerate(duplicate_groups, start=1):
    ordered = sorted(
        paths,
        key=lambda path: (
            split_order.get(path.relative_to(output_dir).parts[0], 99),
            str(path.relative_to(output_dir)),
        ),
    )
    keep_path = ordered[0]
    for duplicate_path in ordered[1:]:
        print(f"Removing duplicate: {duplicate_path.relative_to(output_dir)} (kept {keep_path.relative_to(output_dir)})")
        duplicate_path.unlink()
        removed_count += 1

print(f"\nTotal duplicate files removed: {removed_count}")
if not duplicate_groups:
    print("No byte-identical duplicate images found.")

## Print and display sample images

The output includes each sample's class and filename, followed by a small image grid.

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image

sample_records = random.Random(SEED).sample(
    train_records, k=min(6, len(train_records))
)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax in axes.flat:
    ax.axis("off")

for ax, (image_path, label) in zip(axes.flat, sample_records):
    print(f"Class: {label:10} | File: {image_path.name}")
    with Image.open(image_path) as image:
        ax.imshow(image.convert("RGB"), cmap="gray")
    ax.set_title(label)
    ax.axis("off")

plt.tight_layout()
plt.show()

## Load the folders for a CNN

TensorFlow is included in the install cell near the top of this notebook. If `import tensorflow` previously failed, run that install cell using **Run Cell** (not a terminal command), wait for it to finish, and restart the notebook kernel if VS Code prompts you. Make sure the notebook is using the same Python environment where the install ran. The following creates Keras datasets; it does not train a model.

In [ ]:
import tensorflow as tf

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    output_dir / "train",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    output_dir / "val",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    output_dir / "test",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print("Class order:", train_ds.class_names)